In [ ]:
# Cell 1: Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [ ]:
# Data source: IBM Telco Customer Churn
# Link: https://community.ibm.com/community/user/blogs/steven-macko/2019/07/11/telco-customer-churn-1113
# Download 'Telco_customer_churn.xlsx' and place it in the same directory as this notebook

df = pd.read_excel('Telco_customer_churn.xlsx')
print('Data shape:', df.shape)
df.head(3)

In [ ]:
# Clean Total Charges and drop unneeded/leakage columns
# Justification: The 11 missing values are new customers with 0 tenure months who have not completed a billing cycle yet.
# Setting them to 0.0 reflects that $0.00 has accrued so far, rather than imputing artificial averages.
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce').fillna(0.0)

# Drop customer id, location columns, and post-event leakage columns
cols_to_drop = ['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code', 
                'Lat Long', 'Latitude', 'Longitude', 'Churn Label', 'Churn Score', 
                'CLTV', 'Churn Reason']
df_clean = df.drop(columns=cols_to_drop)

print('Cleaned dataset shape:', df_clean.shape)
print('Any missing values remaining:', df_clean.isnull().sum().sum())
df_clean.head(3)

In [ ]:
# Quick exploration of target variable and contract types
print('Churn Distribution:')
print(df_clean['Churn Value'].value_counts())
print()
print('Churn Rate by Contract:')
print(pd.crosstab(df_clean['Contract'], df_clean['Churn Value'], normalize='index'))

In [ ]:
# Prepare features for Machine Learning and scale numeric columns
X = df_clean.drop(columns=['Churn Value'])
y = df_clean['Churn Value']

# One-hot encode categorical variables
X_encoded = pd.get_dummies(X, drop_first=True)

# Train/Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

# Scale numeric features using StandardScaler to fix convergence warning
num_cols = ['Tenure Months', 'Monthly Charges', 'Total Charges']
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

In [ ]:
# Non-AI Rule-based Baseline:
# Predict churn (1) if customer is on a Month-to-month contract, else (0)
baseline_test_pred = (df_clean.loc[y_test.index, 'Contract'] == 'Month-to-month').astype(int)

base_acc = accuracy_score(y_test, baseline_test_pred)
base_prec = precision_score(y_test, baseline_test_pred)
base_rec = recall_score(y_test, baseline_test_pred)
base_f1 = f1_score(y_test, baseline_test_pred)

print('--- Rule-Based Baseline Results (Test Set) ---')
print(f'Accuracy:  {base_acc:.4f}')
print(f'Precision: {base_prec:.4f}')
print(f'Recall:    {base_rec:.4f}')
print(f'F1-Score:  {base_f1:.4f}')

In [ ]:
# Model 1: Scaled Logistic Regression (default threshold = 0.50)
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

y_probs_lr = lr_model.predict_proba(X_test_scaled)[:, 1]
y_pred_lr = (y_probs_lr >= 0.50).astype(int)

lr_acc = accuracy_score(y_test, y_pred_lr)
lr_prec = precision_score(y_test, y_pred_lr)
lr_rec = recall_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)

print("--- Logistic Regression (threshold = 0.50) ---")
print(f"Accuracy:  {lr_acc:.4f}")
print(f"Precision: {lr_prec:.4f}")
print(f"Recall:    {lr_rec:.4f}")
print(f"F1-Score:  {lr_f1:.4f}")

In [ ]:
# Select decision threshold using 5-fold cross-validation on training data (scaler is refit inside each fold)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold

def make_lr_pipe():
    pre = ColumnTransformer([('num', StandardScaler(), num_cols)], remainder='passthrough')
    return Pipeline([('prep', pre), ('lr', LogisticRegression(max_iter=1000, random_state=42))])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_thresholds = [0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
cv_f1 = {t: [] for t in cv_thresholds}

for tr_idx, val_idx in skf.split(X_train, y_train):   # unscaled X_train; the pipeline scales inside each fold
    pipe = make_lr_pipe().fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
    probs = pipe.predict_proba(X_train.iloc[val_idx])[:, 1]
    for t in cv_thresholds:
        cv_f1[t].append(f1_score(y_train.iloc[val_idx], (probs >= t).astype(int)))

print('--- 5-Fold Cross-Validation on Training Data (Threshold Selection) ---')
for t in cv_thresholds:
    print(f'Threshold {t:.2f} -> Mean CV F1: {np.mean(cv_f1[t]):.4f}')

# 0.30 and 0.35 tie in cross-validation; we use 0.35.
# Now evaluate this fixed choice on the separate holdout test set:
y_pred_lr_tuned = (y_probs_lr >= 0.35).astype(int)

cm_tuned = confusion_matrix(y_test, y_pred_lr_tuned)
print("\n--- Logistic Regression at Fixed 0.35 Threshold (Test Set) ---")
print(f"Accuracy:             {accuracy_score(y_test, y_pred_lr_tuned):.4f}")
print(f"Precision:            {precision_score(y_test, y_pred_lr_tuned):.4f}")
print(f"Recall:               {recall_score(y_test, y_pred_lr_tuned):.4f}")
print(f"F1-Score:             {f1_score(y_test, y_pred_lr_tuned):.4f}")
print(f"Missed Churners (FN): {cm_tuned[1, 0]}")
print(f"False Alarms (FP):    {cm_tuned[0, 1]}")

In [ ]:
# Model 2: Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)

y_probs_rf = rf_model.predict_proba(X_test)[:, 1]
y_pred_rf = (y_probs_rf >= 0.50).astype(int)
y_pred_rf_tuned = (y_probs_rf >= 0.35).astype(int)

rf_acc = accuracy_score(y_test, y_pred_rf)
rf_prec = precision_score(y_test, y_pred_rf)
rf_rec = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)

print("--- Random Forest (threshold = 0.50) ---")
print(f"Accuracy:  {rf_acc:.4f}")
print(f"Precision: {rf_prec:.4f}")
print(f"Recall:    {rf_rec:.4f}")
print(f"F1-Score:  {rf_f1:.4f}")

In [ ]:
# Compare all models on the holdout test set
comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Rule Baseline': [base_acc, base_prec, base_rec, base_f1],
    'Logistic Regression (0.50)': [lr_acc, lr_prec, lr_rec, lr_f1],
    'Logistic Regression (0.35)': [
        accuracy_score(y_test, y_pred_lr_tuned),
        precision_score(y_test, y_pred_lr_tuned),
        recall_score(y_test, y_pred_lr_tuned),
        f1_score(y_test, y_pred_lr_tuned)
    ],
    'Random Forest (0.50)': [rf_acc, rf_prec, rf_rec, rf_f1]
})

print(comparison_df.round(4).to_string(index=False))

In [ ]:
# Figure 1: model comparison and logistic regression threshold tradeoff
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics))
w = 0.2

ax1.bar(x - 1.5*w, comparison_df['Rule Baseline'], w, label='Rule Baseline', color='#7293CB')
ax1.bar(x - 0.5*w, comparison_df['Logistic Regression (0.50)'], w, label='LogReg (0.50)', color='#E1974C')
ax1.bar(x + 0.5*w, comparison_df['Logistic Regression (0.35)'], w, label='LogReg Tuned (0.35)', color='#84BA5B')
ax1.bar(x + 1.5*w, comparison_df['Random Forest (0.50)'], w, label='Random Forest (0.50)', color='#9063CD')

ax1.set_ylabel('Score')
ax1.set_title('(a) Holdout Metric Comparison (n = 1,409)')
ax1.set_xlabel('Metric')
ax1.set_xticks(x)
ax1.set_xticklabels(metrics)
ax1.set_ylim(0, 1.25)
ax1.legend(loc='upper center', ncol=2, fontsize=8.5)
ax1.grid(axis='y', linestyle='--', alpha=0.7)

# Threshold curve
eval_t = np.linspace(0.1, 0.9, 50)
rec_curve = [recall_score(y_test, (y_probs_lr >= th).astype(int)) for th in eval_t]
prec_curve = [precision_score(y_test, (y_probs_lr >= th).astype(int), zero_division=0) for th in eval_t]
f1_curve = [f1_score(y_test, (y_probs_lr >= th).astype(int), zero_division=0) for th in eval_t]

ax2.plot(eval_t, rec_curve, label='Recall', color='#D35E60', linewidth=2)
ax2.plot(eval_t, prec_curve, label='Precision', color='#7293CB', linewidth=2)
ax2.plot(eval_t, f1_curve, label='F1-Score', color='#84BA5B', linewidth=2, linestyle='--')
ax2.axvline(0.50, color='gray', linestyle=':', label='Default Cutoff (0.50)')
ax2.axvline(0.35, color='black', linestyle='-.', label='CV-Tuned Cutoff (0.35)')
ax2.set_title('(b) Logistic Regression Threshold Tradeoff (Test Set)')
ax2.set_xlabel('Threshold')
ax2.set_ylabel('Score')
ax2.set_xlim(0.1, 0.9)
ax2.set_ylim(0, 1.05)
ax2.legend(loc='lower left', fontsize=8.5)
ax2.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig('figure1_results.png', dpi=300)
plt.show()

In [ ]:
# Example output and weak case inspection
# Example 1: Clear prediction
sample_idx = y_test.index[0]
print("--- Example Output (Customer ID:", df.loc[sample_idx, 'CustomerID'], ") ---")
print("Contract:", df.loc[sample_idx, 'Contract'])
print("Tenure:", df.loc[sample_idx, 'Tenure Months'], "months")
print("Monthly Charges: $", df.loc[sample_idx, 'Monthly Charges'])
print("Actual Status:", "Churn" if y_test.loc[sample_idx] == 1 else "Stayed")
print("Predicted Churn Probability:", round(y_probs_lr[0], 4))
print("Prediction at 0.35 threshold:", "Churn" if y_pred_lr_tuned[0] == 1 else "Stayed")

# Example 2: Reconciled Weak Case / False Negative (Customer 3705-PSNGL)
target_cid = '3705-PSNGL'
fn_idx = df[df['CustomerID'] == target_cid].index[0]
fn_pos = list(y_test.index).index(fn_idx)

print("\n--- Weak Case / False Negative (Customer ID:", df.loc[fn_idx, 'CustomerID'], ") ---")
print("Contract:", df.loc[fn_idx, 'Contract'])
print("Tenure:", df.loc[fn_idx, 'Tenure Months'], "months")
print("Monthly Charges: $", df.loc[fn_idx, 'Monthly Charges'])
print("Total Charges: $", df.loc[fn_idx, 'Total Charges'])
print("Observed Exit Reason:", df.loc[fn_idx, 'Churn Reason'])
print("Actual Status: Churn")
print("Predicted Churn Probability:", round(y_probs_lr[fn_pos], 4))
print("Prediction at 0.35 threshold:", "Churn" if y_pred_lr_tuned[fn_pos] == 1 else "Stayed (Missed)")
print("\nDiagnosis Note:")
print("- Observed Reason: Customer departure feedback ('Attitude of support person').")
print("- Cause of Model Error: Static tabular billing data lacks interpersonal service interaction records.")

In [ ]:
# Package versions used for this run
import sys, sklearn
print("Python", sys.version.split()[0], "| scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

In [ ]:
# Table 1: all models at 0.50 and 0.35 on the same 1,409 test customers
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

dt_model = DecisionTreeClassifier(max_depth=5, random_state=42).fit(X_train, y_train)
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42).fit(X_train, y_train)
model_probs = {
    "Decision Tree": dt_model.predict_proba(X_test)[:, 1],
    "Logistic Regression": y_probs_lr,
    "Random Forest": y_probs_rf,
    "Gradient Boosting": gb_model.predict_proba(X_test)[:, 1],
}
rows = []
cmb = confusion_matrix(y_test, baseline_test_pred)
rows.append(["Rule Baseline", "heuristic", accuracy_score(y_test, baseline_test_pred), precision_score(y_test, baseline_test_pred),
             recall_score(y_test, baseline_test_pred), f1_score(y_test, baseline_test_pred),
             roc_auc_score(y_test, baseline_test_pred), cmb[1, 0], cmb[0, 1]])
for name, p in model_probs.items():
    for t in (0.50, 0.35):
        pr = (p >= t).astype(int); cm = confusion_matrix(y_test, pr)
        rows.append([name, t, accuracy_score(y_test, pr), precision_score(y_test, pr), recall_score(y_test, pr),
                     f1_score(y_test, pr), roc_auc_score(y_test, p), cm[1, 0], cm[0, 1]])
table1 = pd.DataFrame(rows, columns=["Model", "Threshold", "Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "Missed (FN)", "False Alarms (FP)"])
print(table1.round(4).to_string(index=False))

In [ ]:
# Figure 2: ROC curves and Random Forest feature importances
from sklearn.metrics import roc_curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
for name, p in model_probs.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    ax1.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc_score(y_test, p):.3f})")
ax1.plot([0, 1], [0, 1], "--", color="gray", label="Random Chance")
ax1.set_xlabel("False Positive Rate"); ax1.set_ylabel("True Positive Rate")
ax1.set_title("(a) ROC Curves on Test Set"); ax1.legend()
importances = pd.Series(rf_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importances.head(10).round(4))
importances.head(8)[::-1].plot.barh(ax=ax2)
ax2.set_title("(b) Top Feature Importances (Random Forest)")
ax2.set_xlabel("Relative Importance (Gini Impurity)")
plt.tight_layout()
plt.savefig("figure2_roc_importance.png", dpi=300)
plt.show()

In [ ]:
# Cost comparison (paper Section 4.3): total cost = C_FN * FN + C_FP * FP, r = C_FN / C_FP
cm50 = confusion_matrix(y_test, (y_probs_lr >= 0.50).astype(int))
cm35 = confusion_matrix(y_test, (y_probs_lr >= 0.35).astype(int))
fn50, fp50, fn35, fp35 = cm50[1, 0], cm50[0, 1], cm35[1, 0], cm35[0, 1]
fnB, fpB = cmb[1, 0], cmb[0, 1]
lower = (fp35 - fp50) / (fn50 - fn35)   # tuned beats default LR when r is above this
upper = (fpB - fp35) / (fn35 - fnB)     # tuned beats the rule baseline when r is below this
print(f"Tuned LR has the lowest total cost when {lower:.2f} < C_FN/C_FP < {upper:.2f}")
for r in (1, 2, 3, 4, 5):
    print(f"r={r}: baseline {r*fnB + fpB} | LR at 0.50: {r*fn50 + fp50} | LR at 0.35: {r*fn35 + fp35}")

In [ ]:
# Paper examples: representative prediction and failure case (logistic regression)
test_ids = df.loc[y_test.index, "CustomerID"].reset_index(drop=True)
for cid in ["4376-KFVRS", "3705-PSNGL"]:
    pos = test_ids[test_ids == cid].index[0]
    row = df[df["CustomerID"] == cid].iloc[0]
    print(cid, "| contract:", row["Contract"], "| tenure:", row["Tenure Months"], "| internet:", row["Internet Service"],
          "| monthly:", row["Monthly Charges"], "| total:", row["Total Charges"], "| actual churn:", row["Churn Value"],
          "| LR churn probability:", round(float(y_probs_lr[pos]), 4))

# Where does 3705-PSNGL rank among the false negatives at 0.35?
fn_mask = (y_test.values == 1) & (y_probs_lr < 0.35)
fn_df = pd.DataFrame({"CustomerID": test_ids[fn_mask].values, "p_churn": y_probs_lr[fn_mask],
                      "Churn Reason": df.loc[y_test.index[fn_mask], "Churn Reason"].values}).sort_values("p_churn").reset_index(drop=True)
print(len(fn_df), "false negatives at 0.35")
print(fn_df.head(8).to_string())
print("Rank of 3705-PSNGL (1 = lowest probability):", fn_df.index[fn_df["CustomerID"] == "3705-PSNGL"][0] + 1)

In [ ]:
# Raw churn rates used in the paper's feature discussion
for col in ["Contract", "Internet Service", "Payment Method", "Dependents"]:
    print(df_clean.groupby(col)["Churn Value"].mean().round(3).to_string(), "\n")
print("Average tenure (months) by churn:")
print(df_clean.groupby("Churn Value")["Tenure Months"].mean().round(1).to_string())
print("\nBaseline false positive rate:", round(cmb[0, 1] / (cmb[0, 0] + cmb[0, 1]), 4))